# 🔄 05 — Convert TransformerLens → HuggingFace Safetensors

**Tujuan:** Convert model dari format internal TransformerLens ke format HuggingFace standard (safetensors) supaya bisa dipakai oleh llama.cpp, Ollama, dan tools lain.

**Lokasi:** VS Code (ringan, reshape tensor saja)

---
**Output:** `models/abliterated/hf_format/` (model + tokenizer dalam safetensors)

Saved results are in `results/reference/`. Your results may differ with other inputs or settings. See `docs/results.md` for details.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src" / "config.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from inside the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from safetensors.torch import load_file
import einops
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

from src.config import DEFAULT_MODEL, ABLITERATED_DIR, ABLITERATED_HF_DIR, DEVICE, SEED

torch.manual_seed(SEED)
ABLITERATED_HF_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output: {ABLITERATED_HF_DIR}")

In [ ]:
# ─── Load state dict yang sudah di-orthogonalize ───
state_dict = load_file(str(ABLITERATED_DIR / 'ablation.safetensors'))
print(f"Loaded {len(state_dict)} tensors")

# Ambil config dari model asli
cfg = AutoConfig.from_pretrained(DEFAULT_MODEL)
print(f"Config: vocab_size={cfg.vocab_size}, hidden_size={cfg.hidden_size}, num_layers={cfg.num_hidden_layers}")

In [ ]:
# --- Convert weight ke format HF ---
# Map dari naming TransformerLens ke HF
# Catatan: Llama 3.2 pake GQA (num_key_value_heads = 8)
# Di HookedTransformer, W_K dan W_V untuk GQA punya prefix underscore: _W_K, _W_V
#
# TL tensor shapes vs HF:
#   W_Q: [n_heads, d_model, d_head] -> q_proj: [n_heads*d_head, d_model]
#   _W_K: [n_kv_heads, d_model, d_head] -> k_proj: [n_kv_heads*d_head, d_model]
#   W_O: [n_heads, d_head, d_model] -> o_proj: [d_model, n_heads*d_head]
#   W_gate: [d_model, intermediate_size] -> gate_proj: [intermediate_size, d_model] (transpose)
#   W_in: [d_model, intermediate_size] -> up_proj: [intermediate_size, d_model] (transpose)
#   W_U: [d_model, vocab_size] -> lm_head: [vocab_size, d_model] (transpose)

hf_state = {}

# Embedding
hf_state['model.embed_tokens.weight'] = state_dict['embed.W_E']

# Per-layer
n_layers = cfg.num_hidden_layers
n_heads = cfg.num_attention_heads
n_kv_heads = cfg.num_key_value_heads

for l in range(n_layers):
    # Attention: TL [n_heads, d_model, d_head] -> HF [n_heads*d_head, d_model]
    w_q = state_dict[f'blocks.{l}.attn.W_Q']
    hf_state[f'model.layers.{l}.self_attn.q_proj.weight'] = einops.rearrange(
        w_q, 'n d h -> (n h) d'
    ).contiguous()

    # _W_K / _W_V: [n_kv_heads, d_model, d_head] -> [n_kv_heads*d_head, d_model]
    w_k = state_dict[f'blocks.{l}.attn._W_K']
    hf_state[f'model.layers.{l}.self_attn.k_proj.weight'] = einops.rearrange(
        w_k, 'n d h -> (n h) d'
    ).contiguous()

    w_v = state_dict[f'blocks.{l}.attn._W_V']
    hf_state[f'model.layers.{l}.self_attn.v_proj.weight'] = einops.rearrange(
        w_v, 'n d h -> (n h) d'
    ).contiguous()

    # W_O: [n_heads, d_head, d_model] -> [d_model, n_heads*d_head]
    w_o = state_dict[f'blocks.{l}.attn.W_O']
    hf_state[f'model.layers.{l}.self_attn.o_proj.weight'] = einops.rearrange(
        w_o, 'n h m -> m (n h)', n=n_heads
    ).contiguous()

    # MLP: TL [d_model, intermediate_size] vs HF [intermediate_size, d_model]
    # Perlu transpose (.T) karena beda convention
    hf_state[f'model.layers.{l}.mlp.gate_proj.weight'] = (
        state_dict[f'blocks.{l}.mlp.W_gate'].T.contiguous()
    )
    hf_state[f'model.layers.{l}.mlp.up_proj.weight'] = (
        state_dict[f'blocks.{l}.mlp.W_in'].T.contiguous()
    )

    # down_proj: TL [intermediate_size, d_model] -> HF [d_model, intermediate_size]
    # Transpose juga
    hf_state[f'model.layers.{l}.mlp.down_proj.weight'] = (
        state_dict[f'blocks.{l}.mlp.W_out'].T.contiguous()
    )

    # Layer norms
    hf_state[f'model.layers.{l}.input_layernorm.weight'] = state_dict[f'blocks.{l}.ln1.w'].contiguous()
    hf_state[f'model.layers.{l}.post_attention_layernorm.weight'] = state_dict[f'blocks.{l}.ln2.w'].contiguous()

# Final norm
hf_state['model.norm.weight'] = state_dict['ln_final.w']

# lm_head: TL [d_model, vocab_size] -> HF [vocab_size, d_model] (transpose)
hf_state['lm_head.weight'] = state_dict['unembed.W_U'].T.contiguous()

print(f'Converted {len(hf_state)} tensors to HF format')


In [ ]:
# ─── Save sebagai safetensors ───
from safetensors.torch import save_file as safetensors_save
safetensors_save(hf_state, str(ABLITERATED_HF_DIR / 'model.safetensors'))
print(f"Saved model weights: {ABLITERATED_HF_DIR / 'model.safetensors'}")

# Config
cfg.save_pretrained(str(ABLITERATED_HF_DIR))
print(f"Saved config: {ABLITERATED_HF_DIR / 'config.json'}")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL)
tokenizer.save_pretrained(str(ABLITERATED_HF_DIR))
print(f"Saved tokenizer: {ABLITERATED_HF_DIR}")

print(f"\nModel siap dipakai di: {ABLITERATED_HF_DIR}")

In [ ]:
# ─── Verifikasi ───
hf_model = AutoModelForCausalLM.from_pretrained(str(ABLITERATED_HF_DIR), torch_dtype=torch.bfloat16)
hf_tokenizer = AutoTokenizer.from_pretrained(str(ABLITERATED_HF_DIR))

# Test generate
test_prompt = "Hello, how are you?"
inputs = hf_tokenizer(test_prompt, return_tensors="pt")
output = hf_model.generate(**inputs, max_new_tokens=20)
response = hf_tokenizer.decode(output[0], skip_special_tokens=True)
print(f"Test output: {response}")
print("HF model berfungsi!")

---
**Next:** Lanjut ke `06_eval_abliterated.ipynb`